# GCP Preflight Check
**Run this before `orchestrate.ipynb`.**

Verifies that every prerequisite is in place:

| # | Check | What it catches |
|---|---|---|
| 1 | Credentials | Env var missing, key file not found |
| 2 | GCS bucket | Bucket doesn't exist or no access |
| 3 | BigQuery dataset | Dataset missing or wrong project |
| 4 | Events table | Not loaded into BigQuery yet |
| 5 | Bronze files — Reddit | Missing JSONL shards per event |
| 6 | Bronze files — Amazon | Missing CSV per event |
| 7 | IAM permissions | Service account lacks required roles |

All checks print `✓ PASS` or `✗ FAIL` with an actionable message. **Do not run `orchestrate.ipynb` until all checks pass.**

---
## Cell 1 — Configuration

In [81]:
import os
from pathlib import Path

import pandas as pd
from google.cloud import bigquery, storage
from google.api_core.exceptions import Forbidden, NotFound

# ── Match these exactly to orchestrate.ipynb ──────────────────────────────────
PROJECT_ID  = "vuthesis-llm-buzz"
BUCKET_NAME = "thesis-bucket-vua"
DATASET_ID  = "thesis_dataset"

PROJECT_ROOT = Path("C:/Users/User/Desktop/VU/Thesis/Code/LLM-prelaunch-buzz-postlaunch-predictor")
EVENTS_PATH  = PROJECT_ROOT / "semi_final_events.csv"
FINAL_EVENTS_PATH = PROJECT_ROOT / "final_events.csv"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"C:\Users\User\.gcp\thesis-sa-key.json"


BQ_DATASET       = f"{PROJECT_ID}.{DATASET_ID}"
BQ_EVENTS_TABLE  = f"{BQ_DATASET}.events"
BQ_BRONZE_REDDIT = f"{BQ_DATASET}.bronze_reddit"
BQ_BRONZE_AMAZON = f"{BQ_DATASET}.bronze_amazon"

# ── Helpers ───────────────────────────────────────────────────────────────────
PASS = "✓ PASS"
FAIL = "✗ FAIL"
WARN = "⚠ WARN"

results = []  # collects (check_name, status, message) for summary

def report(check: str, passed: bool, msg: str, warn: bool = False):
    status = WARN if warn else (PASS if passed else FAIL)
    tag    = f"[{status}]"
    print(f"{tag:<12} {check}")
    if msg:
        print(f"             {msg}")
    results.append((check, status, msg))

print("Configuration loaded. Starting checks...\n")

Configuration loaded. Starting checks...



---
## Check 1 — Credentials

In [82]:
print("═" * 55)
print("CHECK 1 — Credentials")
print("═" * 55)

creds_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")

if not creds_path:
    report(
        "GOOGLE_APPLICATION_CREDENTIALS set", False,
        "Env var not found. Run before starting Jupyter:\n"
        "  set GOOGLE_APPLICATION_CREDENTIALS=C:\\Users\\User\\.gcp\\thesis-sa-key.json"
    )
else:
    report("GOOGLE_APPLICATION_CREDENTIALS set", True, f"→ {creds_path}")

    key_file = Path(creds_path)
    if key_file.exists():
        report("Key file exists", True, f"→ {key_file}")
    else:
        report(
            "Key file exists", False,
            f"File not found at {key_file}. \n"
            "Download it from: GCP Console → IAM → Service Accounts → Keys."
        )

    # Check key file is valid JSON with expected fields
    import json
    try:
        key_data = json.loads(key_file.read_text())
        expected = {"type", "project_id", "private_key", "client_email"}
        missing  = expected - set(key_data.keys())
        if missing:
            report("Key file structure", False, f"Missing fields: {missing}")
        elif key_data.get("project_id") != PROJECT_ID:
            report(
                "Key file project", False,
                f"Key is for project '{key_data.get('project_id')}', "
                f"expected '{PROJECT_ID}'."
            )
        else:
            report("Key file structure", True,
                   f"→ service account: {key_data.get('client_email')}")
    except Exception as e:
        report("Key file structure", False, f"Could not parse key file: {e}")

═══════════════════════════════════════════════════════
CHECK 1 — Credentials
═══════════════════════════════════════════════════════
[✓ PASS]     GOOGLE_APPLICATION_CREDENTIALS set
             → C:\Users\User\.gcp\thesis-sa-key.json
[✓ PASS]     Key file exists
             → C:\Users\User\.gcp\thesis-sa-key.json
[✓ PASS]     Key file structure
             → service account: vuthesis-llm-buzz@vuthesis-llm-buzz.iam.gserviceaccount.com


---
## Check 2 — GCS bucket

In [83]:
print("═" * 55)
print("CHECK 2 — GCS bucket")
print("═" * 55)

try:
    gcs = storage.Client(project=PROJECT_ID)
    bucket = gcs.bucket(BUCKET_NAME)

    if bucket.exists():
        report("Bucket exists", True, f"→ gs://{BUCKET_NAME}")
    else:
        report(
            "Bucket exists", False,
            f"gs://{BUCKET_NAME} not found. Create it in GCP Console → Cloud Storage."
        )

    # Check expected top-level prefixes exist
    for prefix in ["bronze/reddit/", "bronze/amazon/", "logs/"]:
        blobs = list(gcs.list_blobs(BUCKET_NAME, prefix=prefix, max_results=1))
        if blobs:
            report(f"Prefix gs://{BUCKET_NAME}/{prefix}", True, "")
        else:
            report(
                f"Prefix gs://{BUCKET_NAME}/{prefix}", False,
                "No objects found under this prefix — Bronze data may not be collected yet.",
                warn=(prefix == "logs/")  # logs/ missing is a warning, not a blocker
            )

except Forbidden as e:
    report("GCS access", False,
           f"Permission denied. Grant the service account 'Storage Object Admin' role.\n{e}")
except Exception as e:
    report("GCS client init", False, str(e))

═══════════════════════════════════════════════════════
CHECK 2 — GCS bucket
═══════════════════════════════════════════════════════
[✓ PASS]     Bucket exists
             → gs://thesis-bucket-vua
[✓ PASS]     Prefix gs://thesis-bucket-vua/bronze/reddit/
[✓ PASS]     Prefix gs://thesis-bucket-vua/bronze/amazon/
[✓ PASS]     Prefix gs://thesis-bucket-vua/logs/


---
## Check 3 — BigQuery dataset

In [84]:
print("═" * 55)
print("CHECK 3 — BigQuery dataset")
print("═" * 55)

try:
    bq = bigquery.Client(project=PROJECT_ID)

    dataset_ref = bq.dataset(DATASET_ID)
    bq.get_dataset(dataset_ref)
    report("Dataset exists", True, f"→ {BQ_DATASET}")

    # List existing tables
    tables = [t.table_id for t in bq.list_tables(dataset_ref)]
    if tables:
        report("Tables found", True, f"→ {', '.join(sorted(tables))}")
    else:
        report("Tables found", True,
               "Dataset is empty — expected on first run. orchestrate.ipynb will create them.",
               warn=True)

except NotFound:
    report(
        "Dataset exists", False,
        f"Dataset '{DATASET_ID}' not found in project '{PROJECT_ID}'.\n"
        "Create it: GCP Console → BigQuery → + Create Dataset."
    )
except Forbidden as e:
    report("BigQuery access", False,
           f"Permission denied. Grant the service account 'BigQuery Data Editor' + "
           f"'BigQuery Job User' roles.\n{e}")
except Exception as e:
    report("BigQuery client init", False, str(e))

═══════════════════════════════════════════════════════
CHECK 3 — BigQuery dataset
═══════════════════════════════════════════════════════
[✓ PASS]     Dataset exists
             → vuthesis-llm-buzz.thesis_dataset
[✓ PASS]     Tables found
             → events


---
## Check 4 — Events table in BigQuery

The Silver SQL transforms do an `INNER JOIN` against `thesis_dataset.events`. If this table doesn't exist, both Silver SQL files will fail with a `Table not found` error.

If the check fails, run the load command printed in the output.

In [94]:
print("═" * 55)
print("CHECK 4 — Events table in BigQuery")
print("═" * 55)

REQUIRED_BQ_COLS = {"product_event", "launch_date", "product_type", "brand"}

def reload_events_table():
    """
    Load final_events.csv into BigQuery using the Python client.
    Uses WRITE_TRUNCATE so it is safe to re-run at any time.
    """
    import pandas as pd
    from google.cloud.bigquery import LoadJobConfig, WriteDisposition

    df = pd.read_csv(FINAL_EVENTS_PATH)
    job_config = bigquery.LoadJobConfig(
        write_disposition=WriteDisposition.WRITE_TRUNCATE,
        autodetect=True,
    )
    job = bq.load_table_from_dataframe(df, BQ_EVENTS_TABLE, job_config=job_config)
    job.result()
    return bq.get_table(BQ_EVENTS_TABLE)

try:
    events_table = bq.get_table(BQ_EVENTS_TABLE)
    schema_cols  = {f.name for f in events_table.schema}
    missing_cols = REQUIRED_BQ_COLS - schema_cols

    report(
        "events table exists", True,
        f"→ {events_table.num_rows} rows | schema: {sorted(schema_cols)}"
    )

    if missing_cols:
        print(f"             Missing columns {missing_cols} — reloading from {EVENTS_PATH.name}...")
        if not EVENTS_PATH.exists():
            report(
                "events table reload", False,
                f"{EVENTS_PATH.name} not found at {EVENTS_PATH}.\n"
                "Save final_events.csv to the project root first."
            )
        else:
            events_table = reload_events_table()
            schema_cols  = {f.name for f in events_table.schema}
            still_missing = REQUIRED_BQ_COLS - schema_cols
            if still_missing:
                report(
                    "events table reload", False,
                    f"Reload complete but columns still missing: {still_missing}.\n"
                    f"Check that {EVENTS_PATH.name} has headers: {REQUIRED_BQ_COLS}"
                )
            else:
                report(
                    "events table reload", True,
                    f"→ reloaded {events_table.num_rows} rows from {EVENTS_PATH.name} | "
                    f"schema: {sorted(schema_cols)}"
                )
    else:
        report("events table columns", True, "→ all required columns present")

except NotFound:
    print(f"             Table not found — loading from {EVENTS_PATH.name}...")
    if not EVENTS_PATH.exists():
        report(
            "events table exists", False,
            f"Table not found AND {EVENTS_PATH.name} not found at {EVENTS_PATH}.\n"
            "Save final_events.csv to the project root first."
        )
    else:
        events_table = reload_events_table()
        schema_cols  = {f.name for f in events_table.schema}
        missing_cols = REQUIRED_BQ_COLS - schema_cols
        if missing_cols:
            report(
                "events table loaded", False,
                f"Loaded but missing columns: {missing_cols}. "
                f"Check headers in {EVENTS_PATH.name}."
            )
        else:
            report(
                "events table loaded", True,
                f"→ {events_table.num_rows} rows loaded from {EVENTS_PATH.name}"
            )
except Exception as e:
    report("events table check", False, str(e))

═══════════════════════════════════════════════════════
CHECK 4 — Events table in BigQuery
═══════════════════════════════════════════════════════
[✓ PASS]     events table exists
             → 49 rows | schema: ['brand', 'launch_date', 'product_event', 'product_name', 'product_type']
[✓ PASS]     events table columns
             → all required columns present


---
## Check 5 — Bronze files: Reddit

For each event in `events.csv`, checks that at least one JSONL shard exists under `bronze/reddit/{event}/`.

In [88]:
print("═" * 55)
print("CHECK 5 — Bronze files: Reddit")
print("═" * 55)

# Load events.csv locally to get the list of product_events
if not EVENTS_PATH.exists():
    report("events.csv (local)", False,
           f"Not found at {EVENTS_PATH}. Cannot check Bronze coverage.")
else:
    events_df = pd.read_csv(EVENTS_PATH, parse_dates=["launch_date"])
    report("events.csv (local)", True, f"→ {len(events_df)} events loaded")

    print()
    for _, row in events_df.iterrows():
        event  = row["product_event"]
        prefix = f"bronze/reddit/{event}/"
        blobs  = [b for b in gcs.list_blobs(BUCKET_NAME, prefix=prefix)
                  if b.name.endswith(".jsonl")]

        raw_blobs      = [b for b in blobs if "filtered" not in b.name]
        filtered_blobs = [b for b in blobs if "filtered"     in b.name]
        total_bytes    = sum(b.size for b in blobs)

        if not blobs:
            report(
                f"  reddit/{event}", False,
                "No JSONL files found — collection script may not have run for this event."
            )
        else:
            detail = (
                f"{len(raw_blobs)} raw shard(s), "
                f"{len(filtered_blobs)} filtered file(s), "
                f"{total_bytes / 1024:.1f} KB total"
            )
            report(f"  reddit/{event}", True, f"→ {detail}")

═══════════════════════════════════════════════════════
CHECK 5 — Bronze files: Reddit
═══════════════════════════════════════════════════════
[✓ PASS]     events.csv (local)
             → 49 events loaded



C:\Users\User\AppData\Local\Temp\ipykernel_2680\819315513.py:10: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  events_df = pd.read_csv(EVENTS_PATH, parse_dates=["launch_date"])


[✓ PASS]       reddit/iphone_13
             → 45 raw shard(s), 0 filtered file(s), 4939.0 KB total
[✓ PASS]       reddit/iphone_14
             → 158 raw shard(s), 0 filtered file(s), 17865.4 KB total
[✓ PASS]       reddit/iphone_15
             → 20 raw shard(s), 0 filtered file(s), 2262.3 KB total
[✗ FAIL]       reddit/ipad_air_2
             No JSONL files found — collection script may not have run for this event.
[✗ FAIL]       reddit/ipad_7th_gen
             No JSONL files found — collection script may not have run for this event.
[✓ PASS]       reddit/ipad_pro_2018
             → 14 raw shard(s), 0 filtered file(s), 1629.3 KB total
[✗ FAIL]       reddit/ipad_pro_2020
             No JSONL files found — collection script may not have run for this event.
[✓ PASS]       reddit/ipad_pro_m1
             → 16 raw shard(s), 0 filtered file(s), 1972.9 KB total
[✓ PASS]       reddit/ipad_mini_6
             → 24 raw shard(s), 0 filtered file(s), 2692.6 KB total
[✗ FAIL]       reddit/ipa

---
## Check 6 — Bronze files: Amazon

In [90]:
"""

#Parquet to bronze/amazon/{model name} directories in GCS
# ─────────────────────────────────────────────────────────────────────────────
# BRONZE AMAZON SPLITTER
# Reads the two source parquet files from GCS, splits by product model,
# maps to event slugs, and writes one CSV per event to bronze/amazon/{slug}.csv
#
# Run this ONCE before orchestrate.ipynb (or re-run whenever source data changes).
# Safe to re-run — existing CSVs are overwritten, not appended.
# ─────────────────────────────────────────────────────────────────────────────

import hashlib
import io
import json

import google.auth
import pandas as pd
from google.cloud import storage

# ── Config ────────────────────────────────────────────────────────────────────
PROJECT_ID   = "vuthesis-llm-buzz"
BUCKET_NAME  = "thesis-bucket-vua"

# Source parquet files
PARQUET_PHONES      = "amazon_reviews/phones_thesis.parquet"
PARQUET_ELECTRONICS = "amazon_reviews/electronics_thesis.parquet"

# Destination prefix
BRONZE_PREFIX = "bronze/amazon"

# 60-day post-launch window (same as Silver SQL)
WINDOW_MIN = 1
WINDOW_MAX = 60

# ── Model → slug mapping (inverse of SLUG_TO_MODEL) ──────────────────────────
# Key   = exact model string in the parquet files (case-sensitive)
# Value = product_event slug (must match events.csv exactly)
MODEL_TO_SLUG = {
    # iPhones
    "iPhone 13":              "iphone_13",
    "iPhone 14":              "iphone_14",
    "iPhone 15":              "iphone_15",

    # iPads
    "iPad Air 2":             "ipad_air_2",
    "iPad 7th Gen":           "ipad_7th_gen",
    "iPad Pro 2018":          "ipad_pro_2018",
    "iPad Pro 2020":          "ipad_pro_2020",
    "iPad Pro M1":            "ipad_pro_m1",
    "iPad mini 6":            "ipad_mini_6",
    "iPad 9th Gen":           "ipad_9th_gen",
    "iPad Air M1":            "ipad_air_m1",
    "iPad 10th Gen":          "ipad_10th_gen",
    "iPad Pro M2":            "ipad_pro_m2",

    # MacBooks
    "MacBook Air M1":         "macbook_air_m1",
    "MacBook Air M2":         "macbook_air_m2",
    'MacBook Pro 16" Intel':  "macbook_pro_16",

    # Amazon Fire tablets
    "Fire 7 2019":            "fire_7_2019",
    "Fire HD 8 2020":         "fire_hd_8_2020",
    "Fire HD 10 2019":        "fire_hd_10_2019",
    "Fire HD 10 2021":        "fire_hd_10_2021",

    # Other utilitarian
    "Acer Chromebook":        "acer_chromebook",
    "ASUS VivoBook 15":       "asus_vivobook_15",
    "Surface Pro 3":          "surface_pro_3",

    # Motorola
    "Moto G 3rd Gen":         "moto_g_3rd_gen",
    "Moto G 4th Gen":         "moto_g_4th_gen",
    "Moto G Fast":            "moto_g_fast",
    "Moto G Power 2022":      "moto_g_power_2022",

    # Google Pixel
    "Google Pixel 3a":        "pixel_3a",
    "Google Pixel 4a":        "pixel_4a",
    "Google Pixel 4 XL":      "pixel_4_xl",
    "Google Pixel 5":         "pixel_5",
    "Google Pixel 6":         "pixel_6",
    "Google Pixel 7":         "pixel_7",

    # Samsung tablets
    "Galaxy Tab S2":          "galaxy_tab_s2",
    "Galaxy Tab A 2016":      "galaxy_tab_a_2016",
    "Galaxy Tab A 2019":      "galaxy_tab_a_2019",
    "Galaxy Tab A8":          "galaxy_tab_a8",

    # Samsung phones
    "Samsung Galaxy S4":      "galaxy_s4",
    "Samsung Galaxy S5":      "galaxy_s5",
    "Samsung Galaxy S6":      "galaxy_s6",
    "Samsung Galaxy S7 Edge": "galaxy_s7_edge",
    "Samsung Galaxy S8":      "galaxy_s8",
    "Samsung Galaxy S20 FE":  "galaxy_s20_fe",
    "Samsung Galaxy S21":     "galaxy_s21",
    "Samsung Galaxy S21 Ultra":"galaxy_s21_ultra",
    "Samsung Galaxy S22":     "galaxy_s22",
    "Samsung Galaxy S22 Ultra":"galaxy_s22_ultra",
    "Samsung Galaxy S23":     "galaxy_s23",

    # LG
    "LG G3":                  "lg_g3",
}

# ── Column rename: parquet schema → Bronze CSV schema ────────────────────────
RENAME = {
    "text":      "review_text",
    "timestamp": "review_date",
    "model":     "product_name",
}

# ── Auth ──────────────────────────────────────────────────────────────────────
creds, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
gcs = storage.Client(project=PROJECT_ID, credentials=creds)


# ── Helpers ───────────────────────────────────────────────────────────────────

def read_parquet_from_gcs(blob_path: str) -> pd.DataFrame:
    """Download a parquet file from GCS and return as DataFrame."""
    print(f"  Reading gs://{BUCKET_NAME}/{blob_path} ...")
    blob = gcs.bucket(BUCKET_NAME).blob(blob_path)
    buf  = io.BytesIO(blob.download_as_bytes())
    df   = pd.read_parquet(buf)
    print(f"  → {len(df):,} rows, columns: {list(df.columns)}")
    return df


def make_review_id(row) -> str:
    """
    Stable 16-char SHA1 hash from asin + timestamp + text[:50].
    Deterministic — re-running produces identical IDs.
    """
    key = f"{row.get('asin','')}{row.get('timestamp','')}{str(row.get('text',''))[:50]}"
    return hashlib.sha1(key.encode()).hexdigest()[:16]


def transform_to_bronze(df: pd.DataFrame, slug: str) -> pd.DataFrame:
    """Apply window filter, rename columns, add review_id and product_event."""
    # Apply 60-day window
    df = df[df["days_since_launch"].between(WINDOW_MIN, WINDOW_MAX)].copy()

    # Add pipeline fields
    df["review_id"]     = df.apply(make_review_id, axis=1)
    df["product_event"] = slug

    # Rename to Bronze schema
    df = df.rename(columns=RENAME)

    # Normalise review_date to plain date string
    if "review_date" in df.columns:
        df["review_date"] = pd.to_datetime(df["review_date"]).dt.date.astype(str)

    # Select final Bronze columns (graceful — only keeps what exists)
    bronze_cols = [
        "review_id", "asin", "product_name", "product_event",
        "rating", "review_text", "verified_purchase",
        "review_date", "days_since_launch",
    ]
    return df[[c for c in bronze_cols if c in df.columns]]


def upload_bronze_csv(df: pd.DataFrame, slug: str) -> str:
    """Write Bronze CSV to gs://thesis-bucket-vua/bronze/amazon/{slug}.csv"""
    blob_path = f"{BRONZE_PREFIX}/{slug}.csv"
    blob      = gcs.bucket(BUCKET_NAME).blob(blob_path)
    blob.upload_from_string(df.to_csv(index=False), content_type="text/csv")
    return f"gs://{BUCKET_NAME}/{blob_path}"


# ── Main ──────────────────────────────────────────────────────────────────────

print("═" * 65)
print("BRONZE AMAZON SPLITTER")
print("═" * 65)
print()

# Step 1: Load both parquet files and combine
print("Step 1 — Loading source parquet files")
df_phones      = read_parquet_from_gcs(PARQUET_PHONES)
df_electronics = read_parquet_from_gcs(PARQUET_ELECTRONICS)
df_all         = pd.concat([df_phones, df_electronics], ignore_index=True)
print(f"\n  Combined: {len(df_all):,} rows across both files")

# Step 2: Check what model names actually exist in the data
all_models_in_data = set(df_all["model"].unique())
mapped_models      = set(MODEL_TO_SLUG.keys())
unmapped           = all_models_in_data - mapped_models

print(f"\nStep 2 — Model coverage")
print(f"  Models in parquet:     {len(all_models_in_data)}")
print(f"  Models in mapping:     {len(mapped_models & all_models_in_data)}")
if unmapped:
    print(f"  ⚠ Unmapped models ({len(unmapped)}) — skipped:")
    for m in sorted(unmapped):
        n = len(df_all[df_all["model"] == m])
        print(f"    • '{m}'  ({n:,} rows)  ← add to MODEL_TO_SLUG if needed")

# Step 3: Split and upload per event
print(f"\nStep 3 — Splitting and uploading to {BRONZE_PREFIX}/")
print()

results = []

for model_name, slug in sorted(MODEL_TO_SLUG.items(), key=lambda x: x[1]):
    df_model = df_all[df_all["model"] == model_name]

    if df_model.empty:
        print(f"  ⚠ {slug:<30} model '{model_name}' not found in parquet — skipping")
        results.append((slug, model_name, 0, 0, "NOT_FOUND"))
        continue

    total_rows = len(df_model)

    # Transform (includes window filter)
    df_bronze  = transform_to_bronze(df_model, slug)
    window_rows = len(df_bronze)

    if df_bronze.empty:
        print(f"  ⚠ {slug:<30} {total_rows:>5} total | 0 in window — skipping upload")
        results.append((slug, model_name, total_rows, 0, "EMPTY_WINDOW"))
        continue

    # Upload
    try:
        uri = upload_bronze_csv(df_bronze, slug)
        print(f"  ✓ {slug:<30} {total_rows:>5} total | {window_rows:>5} in window → {uri}")
        results.append((slug, model_name, total_rows, window_rows, "OK"))
    except Exception as e:
        print(f"  ✗ {slug:<30} upload failed: {e}")
        results.append((slug, model_name, total_rows, window_rows, "UPLOAD_ERROR"))

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("═" * 65)
print("SUMMARY")
print("═" * 65)

results_df = pd.DataFrame(results,
    columns=["slug", "bq_model", "total_rows", "window_rows", "status"])

ok           = results_df[results_df["status"] == "OK"]
not_found    = results_df[results_df["status"] == "NOT_FOUND"]
empty_window = results_df[results_df["status"] == "EMPTY_WINDOW"]
errors       = results_df[results_df["status"] == "UPLOAD_ERROR"]

print(f"  ✓ Written to bronze/amazon/:  {len(ok)} files")
print(f"  ⚠ Not in parquet:             {len(not_found)} events  ← collect via Amazon notebook")
print(f"  ⚠ Empty after window filter:  {len(empty_window)} events")
print(f"  ✗ Upload errors:              {len(errors)} events")

if not not_found.empty:
    print(f"\n  Events needing Amazon collection:")
    for _, r in not_found.iterrows():
        print(f"    • {r['slug']:<30} (model: '{r['bq_model']}')")

print()

# Styled summary table
def colour_status(val):
    return {
        "OK":           "background-color: #d4edda; color: #155724",
        "NOT_FOUND":    "background-color: #fff3cd; color: #856404",
        "EMPTY_WINDOW": "background-color: #fff3cd; color: #856404",
        "UPLOAD_ERROR": "background-color: #f8d7da; color: #721c24",
    }.get(val, "")

display(
    results_df.style
    .applymap(colour_status, subset=["status"])
    .format({"total_rows": "{:,}", "window_rows": "{:,}"})
    .set_caption("Bronze Amazon splitter results — window_rows = rows written to bronze/amazon/")
)

"""

═════════════════════════════════════════════════════════════════
BRONZE AMAZON SPLITTER
═════════════════════════════════════════════════════════════════

Step 1 — Loading source parquet files
  Reading gs://thesis-bucket-vua/amazon_reviews/phones_thesis.parquet ...
  → 3,887 rows, columns: ['rating', 'text', 'asin', 'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase', 'model', 'type', 'brand', 'launch', 'days_since_launch', 'window_days']
  Reading gs://thesis-bucket-vua/amazon_reviews/electronics_thesis.parquet ...
  → 5,007 rows, columns: ['rating', 'text', 'asin', 'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase', 'model', 'type', 'brand', 'launch', 'days_since_launch', 'window_days']

  Combined: 8,894 rows across both files

Step 2 — Model coverage
  Models in parquet:     35
  Models in mapping:     35

Step 3 — Splitting and uploading to bronze/amazon/

  ✓ acer_chromebook                  124 total |    82 in window → gs://thesis-bucket-vua/bronze/am

C:\Users\User\AppData\Local\Temp\ipykernel_2680\4111791273.py:275: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(colour_status, subset=["status"])


,slug,bq_model,total_rows,window_rows,status
0,acer_chromebook,Acer Chromebook,124,82,OK
1,asus_vivobook_15,ASUS VivoBook 15,162,115,OK
2,fire_7_2019,Fire 7 2019,153,120,OK
3,fire_hd_10_2019,Fire HD 10 2019,0,0,NOT_FOUND
4,fire_hd_10_2021,Fire HD 10 2021,120,77,OK
5,fire_hd_8_2020,Fire HD 8 2020,128,97,OK
6,galaxy_s20_fe,Samsung Galaxy S20 FE,240,140,OK
7,galaxy_s21,Samsung Galaxy S21,0,0,NOT_FOUND
8,galaxy_s21_ultra,Samsung Galaxy S21 Ultra,218,103,OK
9,galaxy_s22,Samsung Galaxy S22,0,0,NOT_FOUND


In [91]:
print("═" * 55)
print("CHECK 6 — Bronze files: Amazon")
print("═" * 55)

for _, row in events_df.iterrows():
    event     = row["product_event"]
    blob_path = f"bronze/amazon/{event}.csv"
    blob      = gcs.bucket(BUCKET_NAME).blob(blob_path)

    if blob.exists():
        blob.reload()
        report(
            f"  amazon/{event}", True,
            f"→ {blob.size / 1024:.1f} KB | updated: {blob.updated.strftime('%Y-%m-%d %H:%M')}"
        )
    else:
        report(
            f"  amazon/{event}", False,
            f"gs://{BUCKET_NAME}/{blob_path} not found. "
            "Run amazon_data_loader.ipynb for this event first."
        )

═══════════════════════════════════════════════════════
CHECK 6 — Bronze files: Amazon
═══════════════════════════════════════════════════════
[✗ FAIL]       amazon/iphone_13
             gs://thesis-bucket-vua/bronze/amazon/iphone_13.csv not found. Run amazon_data_loader.ipynb for this event first.
[✗ FAIL]       amazon/iphone_14
             gs://thesis-bucket-vua/bronze/amazon/iphone_14.csv not found. Run amazon_data_loader.ipynb for this event first.
[✗ FAIL]       amazon/iphone_15
             gs://thesis-bucket-vua/bronze/amazon/iphone_15.csv not found. Run amazon_data_loader.ipynb for this event first.
[✓ PASS]       amazon/ipad_air_2
             → 49.1 KB | updated: 2026-05-21 17:05
[✓ PASS]       amazon/ipad_7th_gen
             → 21.9 KB | updated: 2026-05-21 17:05
[✓ PASS]       amazon/ipad_pro_2018
             → 73.1 KB | updated: 2026-05-21 17:05
[✗ FAIL]       amazon/ipad_pro_2020
             gs://thesis-bucket-vua/bronze/amazon/ipad_pro_2020.csv not found. Run amazon_

---
## Check 7 — IAM permissions

Attempts a lightweight write to GCS and a BigQuery dry-run query to verify the service account has the minimum required permissions without actually modifying anything.

In [92]:
print("═" * 55)
print("CHECK 7 — IAM permissions")
print("═" * 55)

# ── GCS write permission ───────────────────────────────────────────────────────
# Write a tiny test blob, then delete it immediately.
test_blob_path = "logs/.preflight_test"
try:
    test_blob = gcs.bucket(BUCKET_NAME).blob(test_blob_path)
    test_blob.upload_from_string("preflight", content_type="text/plain")
    test_blob.delete()
    report("GCS write (Storage Object Admin)", True, "")
except Forbidden:
    report(
        "GCS write (Storage Object Admin)", False,
        "Service account lacks write access to the bucket.\n"
        "Grant role: Storage Object Admin on gs://" + BUCKET_NAME
    )
except Exception as e:
    report("GCS write test", False, str(e))

# ── BigQuery job permission ────────────────────────────────────────────────────
# Dry-run a trivial SELECT — incurs 0 bytes processed, confirms Job User role.
try:
    job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    bq.query("SELECT 1", job_config=job_config)
    report("BigQuery query (BigQuery Job User)", True, "")
except Forbidden:
    report(
        "BigQuery query (BigQuery Job User)", False,
        "Service account lacks BigQuery Job User role.\n"
        "Grant roles: BigQuery Data Editor + BigQuery Job User."
    )
except Exception as e:
    report("BigQuery query test", False, str(e))

# ── BigQuery table write permission ───────────────────────────────────────────
# Check dataset-level metadata is accessible (proxy for Data Editor access).
try:
    bq.get_dataset(BQ_DATASET)
    report("BigQuery dataset access (BigQuery Data Editor)", True, "")
except Forbidden:
    report(
        "BigQuery dataset access (BigQuery Data Editor)", False,
        f"Cannot access dataset '{BQ_DATASET}'.\n"
        "Grant role: BigQuery Data Editor on this dataset."
    )
except Exception as e:
    report("BigQuery dataset access", False, str(e))

═══════════════════════════════════════════════════════
CHECK 7 — IAM permissions
═══════════════════════════════════════════════════════
[✓ PASS]     GCS write (Storage Object Admin)
[✓ PASS]     BigQuery query (BigQuery Job User)
[✓ PASS]     BigQuery dataset access (BigQuery Data Editor)


---
## Summary

In [93]:
print("═" * 55)
print("PREFLIGHT SUMMARY")
print("═" * 55)

passes   = [r for r in results if r[1] == PASS]
warnings = [r for r in results if r[1] == WARN]
failures = [r for r in results if r[1] == FAIL]

print(f"  {PASS}  {len(passes)} checks passed")
print(f"  {WARN}  {len(warnings)} warnings")
print(f"  {FAIL}  {len(failures)} failures")
print()

if failures:
    print("Failures to fix before running orchestrate.ipynb:")
    for name, status, msg in failures:
        print(f"  • {name}")
        if msg:
            for line in msg.splitlines():
                print(f"      {line}")
    print()
    print("✗  NOT READY — fix the above before proceeding.")
else:
    if warnings:
        print("Warnings (non-blocking):")
        for name, status, msg in warnings:
            print(f"  • {name}: {msg}")
        print()
    print("✓  ALL CHECKS PASSED — safe to run orchestrate.ipynb")

═══════════════════════════════════════════════════════
PREFLIGHT SUMMARY
═══════════════════════════════════════════════════════
  ✓ PASS  38 checks passed
  ⚠ WARN  0 warnings
  ✗ FAIL  14 failures

Failures to fix before running orchestrate.ipynb:
  •   amazon/iphone_13
      gs://thesis-bucket-vua/bronze/amazon/iphone_13.csv not found. Run amazon_data_loader.ipynb for this event first.
  •   amazon/iphone_14
      gs://thesis-bucket-vua/bronze/amazon/iphone_14.csv not found. Run amazon_data_loader.ipynb for this event first.
  •   amazon/iphone_15
      gs://thesis-bucket-vua/bronze/amazon/iphone_15.csv not found. Run amazon_data_loader.ipynb for this event first.
  •   amazon/ipad_pro_2020
      gs://thesis-bucket-vua/bronze/amazon/ipad_pro_2020.csv not found. Run amazon_data_loader.ipynb for this event first.
  •   amazon/macbook_air_m1
      gs://thesis-bucket-vua/bronze/amazon/macbook_air_m1.csv not found. Run amazon_data_loader.ipynb for this event first.
  •   amazon/macbook_

In [99]:
# Full inventory - Amazon + Reddit entries per event

import io

print("═" * 65)
print("CHECK 8 — Bronze corpus coverage per device")
print("═" * 65)
print(f"  Thresholds:  Reddit ≥ {MIN_PRE_LAUNCH} posts  |  Amazon ≥ {MIN_POST_LAUNCH} reviews")
print(f"  Marginal:    Reddit {MIN_PRE_LAUNCH}–{int(MIN_PRE_LAUNCH * 1.2)}         |  Amazon {MIN_POST_LAUNCH}–{int(MIN_POST_LAUNCH * 1.2)}")
print()

MARGINAL_FACTOR = 1.20

def reddit_status(n):
    if n >= MIN_PRE_LAUNCH * MARGINAL_FACTOR: return "PASS",     "✓"
    if n >= MIN_PRE_LAUNCH:                   return "MARGINAL", "⚠"
    return                                           "FAIL",     "✗"

def amazon_status(n):
    if n >= MIN_POST_LAUNCH * MARGINAL_FACTOR: return "PASS",     "✓"
    if n >= MIN_POST_LAUNCH:                   return "MARGINAL", "⚠"
    return                                            "FAIL",     "✗"

# ── Slug → BQ model name mapping ──────────────────────────────────────────────
# Left side must match product_event slugs in events.csv exactly.
# Right side must match the `model` field in BQ exactly (case-sensitive).
SLUG_TO_MODEL = {
 
    # ── Apple — iPhones (electronics_reviews) ─────────────────────────────────
    "iphone_13":        "iPhone 13",
    "iphone_14":        "iPhone 14",
    "iphone_15":        "iPhone 15",           # [VERIFY] — 2023 release, may have limited rows
 
    # ── Apple — iPads (electronics_reviews) ───────────────────────────────────
    "ipad_air_2":       "iPad Air 2",
    "ipad_7th_gen":     "iPad 7th Gen",
    "ipad_pro_2018":    "iPad Pro 2018",
    "ipad_pro_2020":    "iPad Pro 2020",        # [VERIFY]
    "ipad_pro_m1":      "iPad Pro M1",
    "ipad_mini_6":      "iPad mini 6",
    "ipad_9th_gen":     "iPad 9th Gen",
    "ipad_air_m1":      "iPad Air M1",
    "ipad_10th_gen":    "iPad 10th Gen",
    "ipad_pro_m2":      "iPad Pro M2",          # [VERIFY]
 
    # ── Apple — MacBooks (electronics_reviews) ─────────────────────────────────
    "macbook_air_m1":   "MacBook Air M1",       # [VERIFY]
    "macbook_air_m2":   "MacBook Air M2",       # [VERIFY]
    "macbook_pro_16":   "MacBook Pro 16\" Intel",
 
    # ── Amazon Fire tablets (electronics_reviews) ─────────────────────────────
    "fire_7_2019":      "Fire 7 2019",
    "fire_hd_8_2020":   "Fire HD 8 2020",
    "fire_hd_10_2019":  "Fire HD 10 2019",      # [VERIFY] — returned 0 in Check 8
    "fire_hd_10_2021":  "Fire HD 10 2021",
 
    # ── Other utilitarian (electronics_reviews) ───────────────────────────────
    "acer_chromebook":  "Acer Chromebook",      # [VERIFY] — confirm exact BQ string
    "asus_vivobook_15": "ASUS VivoBook 15",     # [VERIFY] — confirm exact BQ string
    "surface_pro_3":    "Surface Pro 3",
 
    # ── Motorola (phones_reviews) ─────────────────────────────────────────────
    "moto_g_3rd_gen":    "Moto G 3rd Gen",
    "moto_g_4th_gen":    "Moto G 4th Gen",
    "moto_g_fast":       "Moto G Fast",
    "moto_g_power_2022": "Moto G Power 2022",
 
    # ── Google Pixel (phones_reviews + electronics_reviews) ───────────────────
    "pixel_3a":  "Google Pixel 3a",
    "pixel_4a":  "Google Pixel 4a",
    "pixel_4_xl":"Google Pixel 4 XL",
    "pixel_5":   "Google Pixel 5",
    "pixel_6":   "Google Pixel 6",
    "pixel_7":   "Google Pixel 7",
 
    # ── Samsung — Tablets (electronics_reviews) ───────────────────────────────
    "galaxy_tab_s2":      "Galaxy Tab S2",
    "galaxy_tab_a_2016":  "Galaxy Tab A 2016",
    "galaxy_tab_a_2019":  "Galaxy Tab A 2019",
    "galaxy_tab_a8":      "Galaxy Tab A8",
 
    # ── Samsung — Flagship phones (phones_reviews + electronics_reviews) ──────
    "galaxy_s4":       "Samsung Galaxy S4",
    "galaxy_s5":       "Samsung Galaxy S5",
    "galaxy_s6":       "Samsung Galaxy S6",
    "galaxy_s7_edge":  "Samsung Galaxy S7 Edge",
    "galaxy_s8":       "Samsung Galaxy S8",
    "galaxy_s20_fe":   "Samsung Galaxy S20 FE",
    "galaxy_s21":      "Samsung Galaxy S21",
    "galaxy_s21_ultra":"Samsung Galaxy S21 Ultra",
    "galaxy_s22":      "Samsung Galaxy S22",
    "galaxy_s22_ultra":"Samsung Galaxy S22 Ultra",
    "galaxy_s23":      "Samsung Galaxy S23",
 
    # ── LG (phones_reviews) ───────────────────────────────────────────────────
    "lg_g3": "LG G3",
}
 
 
# ─────────────────────────────────────────────────────────────────────────────
# DIAGNOSTIC — run this to confirm all [VERIFY] entries exist in BQ
# and to catch any model name mismatches.
# ─────────────────────────────────────────────────────────────────────────────
DIAGNOSTIC_QUERY = """
SELECT model, COUNT(*) AS total_rows,
       COUNTIF(days_since_launch BETWEEN 1 AND 60) AS in_60d_window
FROM (
    SELECT model, days_since_launch FROM `vuthesis-llm-buzz.thesis_data.phones_reviews`
    UNION ALL
    SELECT model, days_since_launch FROM `vuthesis-llm-buzz.thesis_data.electronics_reviews`
)
GROUP BY model
ORDER BY model
"""
 
# Run with:
#   result = bq_client.query(DIAGNOSTIC_QUERY).to_dataframe()
#   print(result.to_string(index=False))
#
# Then compare the `model` column against SLUG_TO_MODEL values.
# Any [VERIFY] entry not appearing in the output means:
#   (a) the model name string is wrong, or
#   (b) the product hasn't been collected yet → run the relevant collection notebook
 
 
# ─────────────────────────────────────────────────────────────────────────────
# CLASSIFICATION FLAGS — review before loading events.csv into BigQuery
# ─────────────────────────────────────────────────────────────────────────────
CLASSIFICATION_FLAGS = {
    "galaxy_s23": (
        "CONFLICT — existing events.csv had Utilitarian; "
        "resolved to Hedonic (Samsung flagship phone, consistent with Galaxy S21/S22). "
        "Update if your thesis framework classifies differently."
    ),
    "macbook_air_m1": (
        "FIXED — existing events.csv had Hedonic; "
        "corrected to Utilitarian (laptop, productivity tool)."
    ),
    "macbook_pro_16": (
        "FIXED — existing events.csv had Hedonic; "
        "corrected to Utilitarian (laptop, productivity tool)."
    ),
    "ipad_pro_m2": (
        "INCONSISTENCY — iPad Pro M1 in new BQ data is classified Hedonic; "
        "iPad Pro M2 (existing) retained as Utilitarian. "
        "Decide: are iPad Pros consistently hedonic or utilitarian across generations?"
    ),
    "acer_chromebook": (
        "NEEDS VERIFICATION — launch date is approximate (2019-06-01). "
        "Confirm exact BQ model name by running DIAGNOSTIC_QUERY."
    ),
    "asus_vivobook_15": (
        "NEEDS VERIFICATION — launch date is approximate (2019-11-01). "
        "Confirm exact BQ model name by running DIAGNOSTIC_QUERY."
    ),
}
 

# ── Pull Amazon 90-day window counts from BigQuery ─────────────────────────────
amazon_query = """
    SELECT model, COUNTIF(days_since_launch BETWEEN 1 AND 90) AS reviews_90d
    FROM (
        SELECT model, days_since_launch FROM `vuthesis-llm-buzz.thesis_data.phones_reviews`
        UNION ALL
        SELECT model, days_since_launch FROM `vuthesis-llm-buzz.thesis_data.electronics_reviews`
    )
    GROUP BY model
"""
amazon_counts = bq.query(amazon_query).to_dataframe()
amazon_lookup = dict(zip(amazon_counts["model"], amazon_counts["reviews_90d"]))

# ── Count Reddit posts from GCS Bronze ────────────────────────────────────────
# ── Count Reddit posts from GCS Bronze (De-duplicated) ───────────────────────
def count_reddit_posts(event: str) -> int:
    # --- Map clean event slugs to list of potential GCS folder names ---
    gcs_folder_mapping = {
    # Apple MacBooks
    "macbook_air_m1":   ["macbook_air_m1_(2020)"],
    "macbook_air_m2":   ["macbook_air_m2"],
    "macbook_pro_16":   ['macbook_pro_16"'],

    # Apple iPads
    "ipad_7th_gen":     ["ipad_7th_generation"],
    "ipad_9th_gen":     ["ipad_9th_generation"],
    "ipad_10th_gen":    ["ipad_10th_generation"],
    "ipad_air_2":       ["ipad_air_2_(2014)"],
    "ipad_air_m1":      ["ipad_air_m1"],
    "ipad_mini_6":      ["ipad_mini_6"],
    "ipad_pro_2018":    ["ipad_pro_2018"],
    "ipad_pro_2020":    ["ipad_pro_2020_"],
    "ipad_pro_m1":      ["ipad_pro_m1"],
    "ipad_pro_m2":      ["ipad_pro_m2"],

    # Apple iPhones
    "iphone_13":        ["iphone_13"],
    "iphone_14":        ["iphone_14"],
    "iphone_15":        ["iphone_15"],

    # Amazon Fire Tablets
    "fire_7_2019":      ["fire_7_(2019)_"],
    "fire_hd_8_2020":   ["fire_hd_8_(2020)"],
    "fire_hd_10_2019":  ["fire_hd_10_(2019)"],
    "fire_hd_10_2021":  ["fire_hd_10_2021"],

    # Google Pixels
    "pixel_3a":         ["pixel_3a",   "google_pixel_3a"],
    "pixel_4a":         ["pixel_4a",   "google_pixel_4a"],
    "pixel_4_xl":       ["pixel_4_xl", "google_pixel_4_xl"],
    "pixel_5":          ["pixel_5"],
    "pixel_6":          ["pixel_6"],
    "pixel_7":          ["pixel_7"],

    # Motorola
    "moto_g_3rd_gen":   ["moto_g_3rd_gen", "moto_g_3rd_gen\t",
                         "motorola_g_3rd_generation"],   # ← all 3 folders
    "moto_g_4th_gen":   ["moto_g_4th_gen", "moto_g4"],  # ← moto_g4 added
    "moto_g_fast":      ["moto_g_fast"],
    "moto_g_power_2022":["moto_g_power_2022"],

    # Samsung Galaxy S Series
    "galaxy_s4":        ["galaxy_s4",        "samsung_galaxy_s4"],
    "galaxy_s5":        ["galaxy_s5",        "samsung_galaxy_s5"],
    "galaxy_s6":        ["galaxy_s6",        "samsung_galaxy_s6"],
    "galaxy_s7_edge":   ["galaxy_s7_edge",   "samsung_galaxy_s7_edge"],
    "galaxy_s8":        ["galaxy_s8",        "samsung_galaxy_s8"],
    "galaxy_s20_fe":    ["galaxy_s20_fe",    "samsung_galaxy_s20_fe"],
    "galaxy_s21":       ["galaxy_s21"],
    "galaxy_s21_ultra": ["galaxy_s21_ultra", "samsung_galaxy_s21_ultra"],  # ← added
    "galaxy_s22":       ["galaxy_s22"],
    "galaxy_s22_ultra": ["galaxy_s22_ultra", "samsung_galaxy_s22_ultra"],  # ← added
    "galaxy_s23":       ["galaxy_s23"],

    # Samsung Galaxy Tablets
    "galaxy_tab_a_2016":["galaxy_tab_a_2016"],
    "galaxy_tab_a_2019":["galaxy_tab_a_2019"],
    "galaxy_tab_a8":    ["galaxy_tab_a8"],
    "galaxy_tab_s2":    ["galaxy_tab_s2"],

    # Other
    "acer_chromebook":  ["acer_chromebook"],
    "asus_vivobook_15": ["asus_vivobook_15"],
    "surface_pro_3":    ["surface_pro_3"],
    "lg_g3":            ["lg_g3"],
}
    
    target_folders = gcs_folder_mapping.get(event, [event])
    if isinstance(target_folders, str):
        target_folders = [target_folders]
    
    unique_post_ids = set()
    fallback_line_count = 0
    has_json_structure = True

    # Scan across every associated folder path
    for folder in target_folders:
        prefix = f"bronze/reddit/{folder}/"
        blobs  = [b for b in gcs.list_blobs(BUCKET_NAME, prefix=prefix)
                  if b.name.endswith(".jsonl") and "filtered" not in b.name]
                  
        for blob in blobs:
            content = blob.download_as_text(encoding="utf-8")
            for line in content.splitlines():
                if not line.strip():
                    continue
                
                fallback_line_count += 1
                
                # Extract unique IDs to make sure we don't double-count overlapping logs
                if has_json_structure:
                    try:
                        post_data = json.loads(line)
                        post_id = post_data.get("id") or post_data.get("name") or hash(line)
                        unique_post_ids.add(post_id)
                    except Exception:
                        has_json_structure = False

    return len(unique_post_ids) if has_json_structure else fallback_line_count

# ── Run per event ──────────────────────────────────────────────────────────────
rows = []
for _, row in events_df.iterrows():
    event = row["product_event"]
    ptype = "hedonic" if row["product_type"] == 1 else "utilitarian"

    r_count = count_reddit_posts(event)

    bq_model = SLUG_TO_MODEL.get(event)
    a_count  = amazon_lookup.get(bq_model, 0) if bq_model else 0
    a_note   = "" if bq_model else " ⚠ no BQ mapping"

    r_label, r_icon = reddit_status(r_count)
    a_label, a_icon = amazon_status(a_count)

    rows.append({
        "product_event":  event,
        "bq_model":       bq_model or "— unmapped —",
        "type":           ptype,
        "reddit_n":       r_count,
        "reddit_status":  r_label,
        "amazon_90d":     a_count,
        "amazon_status":  a_label,
        "both_pass":      r_label != "FAIL" and a_label != "FAIL",
    })

    print(
        f"  {event:<28} [{ptype:<11}]  "
        f"Reddit: {r_count:>6}  {r_icon} {r_label:<8}  |  "
        f"Amazon: {a_count:>6}  {a_icon} {a_label}{a_note}"
    )

# ── Summary ────────────────────────────────────────────────────────────────────
coverage_df = pd.DataFrame(rows)
both_pass   = coverage_df[coverage_df["both_pass"]]

print(f"\n{'═'*65}")
print(f"  Events with BOTH Reddit and Amazon passing: {len(both_pass)}")
print(f"{'═'*65}")
for _, r in both_pass.iterrows():
    print(f"  ✓  {r['product_event']:<28} [{r['type']:<11}]  "
          f"Reddit: {r['reddit_n']:>6}  |  Amazon: {r['amazon_90d']:>6}")

# ── Styled table ───────────────────────────────────────────────────────────────
def colour_status(val):
    return {
        "PASS":     "background-color: #d4edda; color: #155724",
        "MARGINAL": "background-color: #fff3cd; color: #856404",
        "FAIL":     "background-color: #f8d7da; color: #721c24",
    }.get(val, "")

display(
    coverage_df[["product_event","type","reddit_n","reddit_status","amazon_90d","amazon_status"]]
    .style
    .applymap(colour_status, subset=["reddit_status","amazon_status"])
    .format({"reddit_n": "{:,}", "amazon_90d": "{:,"})
    .set_caption("Coverage check — Amazon counts from BigQuery 90-day window")
)

═════════════════════════════════════════════════════════════════
CHECK 8 — Bronze corpus coverage per device
═════════════════════════════════════════════════════════════════
  Thresholds:  Reddit ≥ 500 posts  |  Amazon ≥ 200 reviews
  Marginal:    Reddit 500–600         |  Amazon 200–240



c:\Users\User\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  iphone_13                    [hedonic    ]  Reddit:   9687  ✓ PASS      |  Amazon:      0  ✗ FAIL
  iphone_14                    [hedonic    ]  Reddit:  16134  ✓ PASS      |  Amazon:      0  ✗ FAIL
  iphone_15                    [hedonic    ]  Reddit:   1909  ✓ PASS      |  Amazon:      0  ✗ FAIL
  ipad_air_2                   [hedonic    ]  Reddit:   2084  ✓ PASS      |  Amazon:    254  ✓ PASS
  ipad_7th_gen                 [hedonic    ]  Reddit:    149  ✗ FAIL      |  Amazon:    292  ✓ PASS
  ipad_pro_2018                [hedonic    ]  Reddit:   3443  ✓ PASS      |  Amazon:    371  ✓ PASS
  ipad_pro_2020                [hedonic    ]  Reddit:   2932  ✓ PASS      |  Amazon:      0  ✗ FAIL
  ipad_pro_m1                  [hedonic    ]  Reddit:   4706  ✓ PASS      |  Amazon:    184  ✗ FAIL
  ipad_mini_6                  [hedonic    ]  Reddit:   1803  ✓ PASS      |  Amazon:    197  ✗ FAIL
  ipad_9th_gen                 [hedonic    ]  Reddit:     33  ✗ FAIL      |  Amazon:    185  ✗ FAIL


C:\Users\User\AppData\Local\Temp\ipykernel_2680\385497983.py:333: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(colour_status, subset=["reddit_status","amazon_status"])


ValueError: unmatched '{' in format spec

In [98]:
# Folder names inside the bronze/ directory (used for slug_to_map and debugging)
TARGET_FOLDER = "bronze/reddit/"

print(f"Scanning '{BUCKET_NAME}/{TARGET_FOLDER}' for subfolders...")

# prefix restricts the search to this folder
# delimiter='/' ensures we only see the immediate next level of folders
blobs = gcs.list_blobs(BUCKET_NAME, prefix=TARGET_FOLDER, delimiter="/")

# Trigger the API request to populate the prefixes attribute
list(blobs)

if blobs.prefixes:
    print(f"\nFound folders inside {TARGET_FOLDER}:")
    for subfolder in blobs.prefixes:
        print(f"📁 {subfolder}")
else:
    print(f"❌ No subfolders found inside '{TARGET_FOLDER}'!")

Scanning 'thesis-bucket-vua/bronze/reddit/' for subfolders...

Found folders inside bronze/reddit/:
📁 bronze/reddit/galaxy_s20_fe/
📁 bronze/reddit/galaxy_tab_a_2016/
📁 bronze/reddit/iphone_15/
📁 bronze/reddit/ipad_10th_gen/
📁 bronze/reddit/moto_g_3rd_gen	/
📁 bronze/reddit/galaxy_s5/
📁 bronze/reddit/moto_g_3rd_gen/
📁 bronze/reddit/motorola_g_3rd_generation/
📁 bronze/reddit/ipad_pro_2020_/
📁 bronze/reddit/ipad_9th_gen/
📁 bronze/reddit/samsung_galaxy_s21_ultra/
📁 bronze/reddit/surface_pro_3/
📁 bronze/reddit/macbook_air_m2/
📁 bronze/reddit/galaxy_s23/
📁 bronze/reddit/asus_vivobook_15/
📁 bronze/reddit/ipad_10th_generation/
📁 bronze/reddit/pixel_7/
📁 bronze/reddit/pixel_6/
📁 bronze/reddit/galaxy_tab_a_2019/
📁 bronze/reddit/galaxy_tab_a8/
📁 bronze/reddit/samsung_galaxy_s4/
📁 bronze/reddit/moto_g4/
📁 bronze/reddit/macbook_air_m1_(2020)/
📁 bronze/reddit/moto_g_power_2022/
📁 bronze/reddit/ipad_pro_2018/
📁 bronze/reddit/pixel_3a/
📁 bronze/reddit/samsung_galaxy_s5/
📁 bronze/reddit/ipad_9th_generat